## Affine-Partitioned Neural Networks (AP-NNs)

Self-contained: the closed-form GS pricer is copied in below (not imported from `gs_wamol`), with an autodiff PDE self-check.

Architecture:
- **Path net** `delta_hat_phi(t)` — latent convenience yield.
- **Drift net** `A_hat_theta(tau)` — `B(tau)` is analytic (closed form); only `A(tau)` is learned.
- **Pricing transform**: `F_hat = S * exp(B(tau) * delta_hat + A_hat)`.
- **Loss** = `e_data` (price MSE) + `e_ode` (ODE residual on `A_hat`) + `e_sde` (OU transition NLL on the path), WamOL-balanced.

In [ ]:
import os
import sys
import types
import pickle
import time
from dataclasses import dataclass, asdict
from typing import Callable, Dict, Tuple, Union

import jax
import jax.numpy as jnp
from jax import grad, jacrev, jit, lax, random, vmap
from jax.nn.initializers import glorot_normal, normal, zeros
from jax.flatten_util import ravel_pytree
from jax.tree_util import tree_leaves, tree_map

import optax
import ml_collections
from flax import linen as nn
from flax.training import train_state, orbax_utils
import orbax.checkpoint

import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
print("JAX:", jax.__version__, "| x64 enabled:", jax.config.jax_enable_x64,
      "| default dtype:", jnp.ones(1).dtype)

### Closed-form Gibson–Schwartz pricer

$$\hat{F}(\tau, S, \delta) = S\exp\!\big[B(\tau)\delta + A(\tau)\big],\qquad B(\tau) = -\frac{1-e^{-\kappa\tau}}{\kappa}$$

`B_coeff` is used directly in the pricing transform below. `A_coeff`/`futures`/`GSParams` are not used in the loss (that would trivially satisfy the PDE) — the PDE self-check itself lives in `tests/test_synthetic_gs.py`.

In [ ]:
# Do not edit these coefficients without re-running tests/test_synthetic_gs.py (PDE self-check verified).
@dataclass(frozen=True)
class GSParams:
    """Risk-neutral GS Model 2 params. alpha_Q = alpha_P - sigma2*lambda2/kappa."""
    r: float
    kappa: float
    alpha_Q: float
    sigma1: float
    sigma2: float
    rho: float


def B_coeff(tau, kappa):
    return -(1.0 - jnp.exp(-kappa * tau)) / kappa


def A_coeff(tau, p: GSParams):
    kappa = p.kappa
    s1s2rho = p.sigma1 * p.sigma2 * p.rho
    s2sq = p.sigma2 ** 2

    lin = (p.r - p.alpha_Q + 0.5 * s2sq / kappa ** 2 - s1s2rho / kappa) * tau
    two_kappa = 0.25 * s2sq * (1.0 - jnp.exp(-2.0 * kappa * tau)) / kappa ** 3
    one_kappa = (p.alpha_Q * kappa + s1s2rho - s2sq / kappa) * (1.0 - jnp.exp(-kappa * tau)) / kappa ** 2
    return lin + two_kappa + one_kappa


def log_futures(tau, S, delta, p: GSParams):
    return jnp.log(S) + B_coeff(tau, p.kappa) * delta + A_coeff(tau, p)


def futures(tau, S, delta, p: GSParams):
    return jnp.exp(log_futures(tau, S, delta, p))


# Pickle shim: mc_data.pkl stores GSParams by class path; resolve to the class above.
_gs_shim = types.ModuleType("gs_wamol.physics.gibson_schwartz")
_gs_shim.GSParams = GSParams
sys.modules.setdefault("gs_wamol", types.ModuleType("gs_wamol"))
sys.modules.setdefault("gs_wamol.physics", types.ModuleType("gs_wamol.physics"))
sys.modules["gs_wamol.physics.gibson_schwartz"] = _gs_shim

### Network architecture

Generic `MLP` with optional weight-factorization reparameterisation on `Dense`. Two instances are built below: the drift net `A_hat_theta(tau)` and the path net `delta_hat_phi(t)`.

In [ ]:
activation_fn = {
    "tanh": jnp.tanh,
    "sin": jnp.sin,
}


def _get_activation(name):
    if name in activation_fn:
        return activation_fn[name]
    raise NotImplementedError(f"Activation {name} not in dictionary: {activation_fn.keys()}")


def _weight_fact(init_fn, mean, stddev):
    def init(key, shape):
        key1, key2 = random.split(key)
        w = init_fn(key1, shape)
        g = mean + normal(stddev)(key2, (shape[-1],))
        g = jnp.exp(g)
        v = w / g
        return g, v
    return init


class Dense(nn.Module):
    features: int
    kernel_init: Callable = glorot_normal()
    bias_init: Callable = zeros
    reparam: Union[None, Dict] = None

    @nn.compact
    def __call__(self, x):
        if self.reparam is None:
            kernel = self.param(
                "kernel", self.kernel_init, (x.shape[-1], self.features)
            )
        elif self.reparam["type"] == "weight_fact":
            g, v = self.param(
                "kernel",
                _weight_fact(
                    self.kernel_init,
                    mean=self.reparam["mean"],
                    stddev=self.reparam["stddev"],
                ),
                (x.shape[-1], self.features),
            )
            kernel = g * v
        bias = self.param("bias", self.bias_init, (self.features,))
        y = jnp.dot(x, kernel) + bias
        return y


class MLP(nn.Module):
    hidden_dim: Tuple[int] = (32, 16)
    out_dim: int = 1
    activation: str = "tanh"
    reparam: Union[None, Dict] = None

    def setup(self):
        self.activation_fn = _get_activation(self.activation)

    @nn.compact
    def __call__(self, x):
        for i in range(len(self.hidden_dim)):
            x = Dense(features=self.hidden_dim[i], reparam=self.reparam)(x)
            x = self.activation_fn(x)
        x = Dense(features=self.out_dim, reparam=self.reparam)(x)
        return x

### Build Networks

* Path net: `delta_hat_phi(t)` — latent convenience yield.
* Drift net: `A_hat_theta(tau)` — `B(tau)` is analytic (closed form); only `A(tau)` is learned.
* Pricing transform: `F_hat = S * exp(B(tau) * delta_hat + A_hat)`.
* Loss = `e_data` (price MSE) + `e_ode` (ODE residual on `A_hat`) + `e_sde` (OU transition NLL on the path), WamOL-balanced.

```mermaid
flowchart LR
    tau --> normalize_tau --> drift_ann --> A_hat["A_hat(tau) = tau * N_theta(tau)"]
    t --> normalize_time --> path_ann --> delta_hat["delta_hat_phi(t)"]
    A_hat --> logF["log F_hat = log(S) + B(tau)*delta + A_hat(tau)"]
    delta_hat --> logF
    S --> logF
```


In [ ]:
def build_nets(config):
    reparam = None
    if config.ann_reparam:
        reparam = ml_collections.ConfigDict({"type": "weight_fact", "mean": 0.5, "stddev": 0.1})

    # drift_ann takes tau alone (in_dim 1), not (S, delta, tau) (in_dim 3).
    drift_ann = MLP(hidden_dim=config.drift_hidden_dim,
                    out_dim=1,
                    activation=config.ann_activation_str,
                    reparam=reparam)
    path_ann = MLP(hidden_dim=config.path_hidden_dim,
                   out_dim=1,
                   activation=config.ann_activation_str,
                   reparam=reparam)
    return drift_ann, path_ann


# psi = {kappa, sigma1, sigma2, rho, alpha_Q, alpha_P}; kappa/sigma2 are shared across
# Q/P (measure-invariant); softplus/tanh reparam keeps the optimiser unconstrained.
R_FIXED = 0.05


def _softplus_inv(y):
    # invert softplus so a desired positive start value maps to its raw pre-image
    return float(np.log(np.expm1(y)))


# one (forward, inverse) pair per psi parameter -- constrain_psi/raw_psi_init below
# are both derived from this single table instead of independently retyping each
# parameter's transform twice (forward direction and its inverse).
PSI_TRANSFORMS = {
    "kappa":   (jax.nn.softplus, _softplus_inv),
    "sigma1":  (jax.nn.softplus, _softplus_inv),
    "sigma2":  (jax.nn.softplus, _softplus_inv),
    "rho":     (jnp.tanh, lambda y: float(np.arctanh(y))),
    "alpha_Q": (lambda x: x, lambda y: y),
    "alpha_P": (lambda x: x, lambda y: y),
}


def constrain_psi(raw):
    return {k: fwd(raw[k]) for k, (fwd, _) in PSI_TRANSFORMS.items()}


def raw_psi_init(psi0):
    # psi0: dict of desired *constrained* start values -> raw (unconstrained) leaf
    return {k: jnp.asarray(inv(psi0[k])) for k, (_, inv) in PSI_TRANSFORMS.items()}


def psi_to_gsparams(psi):
    # Effective Q pricing params + the P transition triple, all from learned psi.
    p_Q_eff = GSParams(r=R_FIXED, kappa=psi["kappa"], alpha_Q=psi["alpha_Q"],
                       sigma1=psi["sigma1"], sigma2=psi["sigma2"], rho=psi["rho"])
    return {"p_Q": p_Q_eff, "kappa_P": psi["kappa"],
            "alpha_P": psi["alpha_P"], "sigma2_P": psi["sigma2"]}


def make_fns(drift_ann, path_ann, params):
    # psi is a learned leaf of params; gsp (learned, not true) is returned for error()/plots.
    psi = constrain_psi(params["psi"])
    gsp = psi_to_gsparams(psi)
    p_Q_eff = gsp["p_Q"]

    # boundary condition A(0)=0 -> tau * N(0) = 0 regardless of N's output (hard)
    def A_hat_fn(tau):
        return tau * drift_ann.apply(params["drift"], normalize_tau(tau))[0]

    # pricing transform: spot, analytic B(tau; learned kappa), state delta, learned A(tau)
    def log_F_hat_fn(S, delta, tau):
        return jnp.log(S) + B_coeff(tau, p_Q_eff.kappa) * delta + A_hat_fn(tau)

    def F_hat_fn(S, delta, tau):
        return jnp.exp(log_F_hat_fn(S, delta, tau))

    def delta_hat_fn(t):
        return path_ann.apply(params["path"], normalize_time(t))[0]

    return F_hat_fn, delta_hat_fn, A_hat_fn, gsp


# one param dict so optax/Flax jointly train the drift net, path net AND psi.
def init_nets(config, key):
    drift_ann, path_ann = build_nets(config)
    k_s, k_p = random.split(key, 2)
    params = {
        "drift": drift_ann.init(k_s, jnp.ones((1,))),
        "path": path_ann.init(k_p, jnp.ones((1,))),
        "psi": raw_psi_init(PSI_INIT),   # perturbed-from-truth init (see data cell)
    }
    return drift_ann, path_ann, params

### Optimiser

Core code inherited from https://github.com/khoshisashi/AC-PINNs/blob/main/AC_PINNs_IVS_calibration.ipynb for the original SABR / Dupire-local-vol problem by Khoshisashi et al. 2023. in his ACPINN repo.

In [ ]:
import json

with open("../config/hp.sweep.json", "r") as f:
    adam_hp = json.load(f)

# adam with exponential LR decay: large steps early, smaller as it settles near a minimum
def make_optimizer(hp):
    lr = optax.exponential_decay(
        init_value=hp["learning_rate"], 
        transition_steps=hp["decay_steps"],
        decay_rate=hp["decay_rate"],
    )
    return optax.adam(
        learning_rate=lr, b1=hp["beta1"], b2=hp["beta2"], eps=hp["adam_eps"]
    )

### Input Data

Monte Carlo simulated data generated by `monte_carlo.ipynb` and saved to `data/gs_mc_data.npz`.

Split across the next two cells so the source is swappable: the first builds `data`/`taus`/`mesh` (all `calibration`/`error` consume — a real-WTI loader is a drop-in here); the second holds `delta_true`/`PSI_TRUE`, MC-only ground truth with no real-data equivalent.

In [ ]:
import pandas as pd

_MC_DATA_PATH = "../data/input/synthetic/mc_data.pkl"
_WTI_DATA_PATH = "../data/input/real/wti_analysis_ready.csv"

def get_data_mc(path_id=0, path=_MC_DATA_PATH):
    """MC loader: bundles generic (t, S, taus, log_F_obs) with MC-only ground truth."""
    with open(path, "rb") as f:
        mc_data = pickle.load(f)

    taus = mc_data["taus"]                             
    t_train = mc_data["t_grid"]                          
    S_train = mc_data["S"][path_id]                      
    log_F_obs_train = mc_data["log_F_obs"][path_id]

    delta_true = mc_data["delta_true"][path_id] # eval target           

    p_Q = GSParams(**asdict(mc_data["params_Q"]))
    kappa_P = mc_data["params_P"]["kappa"]                 
    alpha_P = mc_data["params_P"]["alpha_P"]               
    sigma2_P = mc_data["params_P"]["sigma2"]               

    return t_train, S_train, taus, log_F_obs_train, delta_true, p_Q, kappa_P, alpha_P, sigma2_P

def get_data_wti(path=_WTI_DATA_PATH, n_maturities=8):
    """WTI loader: long (date x contract) panel -> rectangular (dates x n_maturities).

    Ranks each date's live contracts by tau ascending into maturity slots
    0..n_maturities-1, keeping only dates with >= n_maturities contracts
    (n_maturities=8 keeps ~96% of dates -- 8 is the modal contract count;
    the rest have 6-7 and are dropped). `taus` is the per-slot MEAN tau across
    all kept dates -- a static stand-in for what is actually a per-date
    shrinking tau. error() only accepts one shared taus vector today; feeding
    it the true per-date tau matrix instead of this mean requires updating
    error() itself, which hasn't been done yet.
    """
    df = pd.read_csv(path)
    df = df.dropna(subset=["spot", "rate"])  # drops 2020-04-20 (negative-price day)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["date", "tau"])
    df["slot"] = df.groupby("date").cumcount()
    df = df[df["slot"] < n_maturities]
    df = df[df.groupby("date")["slot"].transform("count") == n_maturities]

    tau_matrix = df.pivot(index="date", columns="slot", values="tau").sort_index()
    settle_matrix = df.pivot(index="date", columns="slot", values="settle").sort_index()
    spot_by_date = df.groupby("date")["spot"].first().sort_index()
    rate_by_date = df.groupby("date")["rate"].first().sort_index()

    dates = tau_matrix.index.to_numpy()
    t_train = ((dates - dates[0]) / np.timedelta64(365, "D")).astype(float)
    S_train = spot_by_date.to_numpy()
    taus = tau_matrix.to_numpy().mean(axis=0)
    log_F_obs_train = np.log(settle_matrix.to_numpy())
    r_train = rate_by_date.to_numpy()

    delta_true = p_Q = kappa_P = alpha_P = sigma2_P = None  # no ground truth for real data

    return t_train, S_train, taus, log_F_obs_train, delta_true, p_Q, kappa_P, alpha_P, sigma2_P, r_train


In [ ]:
t_train, S_train, taus, log_F_obs_train, delta_true, p_Q, kappa_P, alpha_P, sigma2_P = get_data_mc(path_id=0)

# data/taus/mesh/normalize_* are ALL that calibration()/error() consume -- source-agnostic
data = (t_train, S_train, log_F_obs_train)


T_SCALE = float(t_train[-1])
TAU_SCALE = float(taus.max())


def normalize_tau(tau):
    return jnp.atleast_1d(tau / TAU_SCALE) # ensure output shape 1-D


def normalize_time(t):
    return jnp.atleast_1d(t / T_SCALE)


# collocation mesh to evaluate physics residuals at -> random tau values spread across the mesh [0, tau_max] -> sampled from
N_COLLOCATION = 4096

_key_mesh = random.PRNGKey(0) # evaluates at diff points each run
tau_mesh = random.uniform(_key_mesh, (N_COLLOCATION,), minval=0.0, maxval=float(taus.max()))
mesh = tau_mesh

print("t_train        ", t_train.shape, f"[{float(t_train.min()):.2f}, {float(t_train.max()):.2f}]")
print("S_train        ", S_train.shape, f"[{float(S_train.min()):.2f}, {float(S_train.max()):.2f}]")
print("taus           ", taus.shape, f"[{float(taus.min()):.3f}, {float(taus.max()):.3f}]")
print("log_F_obs_train", log_F_obs_train.shape, "(dates x maturities)")
print("collocation    ", tau_mesh.shape, "points over tau")

In [ ]:
# MC-only ground truth: real data has no delta_true/PSI_TRUE (delta_t is latent in reality).
# PSI_INIT is perturbed FROM PSI_TRUE (~30-50% off) so training is a recovery test, not
# stability check; a real-data run would need PSI_INIT set independently instead.
def psi_dict(p_Q, alpha_P):
    """GSParams + alpha_P -> the canonical 6-key psi dict (shared by PSI_TRUE here
    and by the recovered psi_hat in the path-repeat validation cell)."""
    return {
        "kappa":   float(p_Q.kappa),
        "sigma1":  float(p_Q.sigma1),
        "sigma2":  float(p_Q.sigma2),
        "rho":     float(p_Q.rho),
        "alpha_Q": float(p_Q.alpha_Q),
        "alpha_P": float(alpha_P),
    }


PSI_TRUE = psi_dict(p_Q, alpha_P)
PSI_INIT = {
    "kappa":   PSI_TRUE["kappa"]  * 1.5,
    "sigma1":  PSI_TRUE["sigma1"] * 0.7,
    "sigma2":  PSI_TRUE["sigma2"] * 1.3,
    "rho":     PSI_TRUE["rho"]    * 0.6,
    "alpha_Q": PSI_TRUE["alpha_Q"] + 0.06,
    "alpha_P": PSI_TRUE["alpha_P"] - 0.06,
}

print("delta_true     ", delta_true.shape, "HELD OUT -- evaluation only")
print("PSI_TRUE       ", {k: round(v, 4) for k, v in PSI_TRUE.items()})
print("PSI_INIT       ", {k: round(v, 4) for k, v in PSI_INIT.items()}, "(learned from here)")

### Loss terms — `error()`

Rewritten to match the attached architecture diagram (drift net + analytic B(tau) + WamOL-balanced loss channels), replacing the earlier terms from `docs/mathematical_derivations-5.pdf`.

- `e_data` = the diagram's $\hat{\mathcal{L}}_{\text{data}}$ — data misfit in log-price space (was `e_acc`)
- `e_ode` = the diagram's $\hat{\mathcal{L}}_{\text{ODE}}$ — ODE residual on $\hat{A}(\tau)$ (replaces the old 3-D PDE residual `e_pde`)
- `e_sde` = the diagram's $\hat{\mathcal{L}}_{\text{SDE}}$ — OU transition-density residual on the recovered path
- `e_cac`/`e_rcac` = `mathematical_derivations-6.pdf` §7.1-7.2 — cash-and-carry ceiling / reverse-cash-and-carry floor, hinge-squared, evaluated on the data manifold `(S_i, delta_hat(t_i), tau_i)` (not the PDE mesh — the ceiling doesn't reference `delta` and would fight the PDE residual off-manifold)
- `e_delta_floor` (§7.3) and $\mathcal{L}_b$ remain **not implemented** — out of scope for now

In [ ]:
STORAGE_COST_U = 0.5   # per-unit-time storage cost; not fixed by the derivation, default 0
DELTA_MAX = 0.75       # rcac wedge; not fixed by the derivation either -- set here as the
                       # empirical upper edge of delta_true across the 100 MC paths (~0.75
                       # at the 99th percentile), mirroring how delta_min=-0.3 was set in
                       # eq. 39. A placeholder plausibility ceiling, not a derived constant.


def error(fns, data, mesh):
    F_hat_fn, delta_hat_fn, A_hat_fn, gsp = fns
    # gsp holds the learned (not true) effective Q params + P transition triple
    p_Q = gsp["p_Q"]
    kappa_P = gsp["kappa_P"]
    alpha_P = gsp["alpha_P"]
    sigma2_P = gsp["sigma2_P"]
    t_train, S_train, log_F_obs_train = data
    tau_mesh = mesh

    # Latent path at observation times. Input is t alone (S6 remark 1).
    delta_hat = vmap(delta_hat_fn)(t_train)                          # (n,)

    def log_F_hat_fn(S, delta, tau):
        return jnp.log(S) + B_coeff(tau, p_Q.kappa) * delta + A_hat_fn(tau)

    log_F_hat = vmap(
        lambda S, d: vmap(lambda tau: log_F_hat_fn(S, d, tau))(taus)
    )(S_train, delta_hat)                                            # (n, K)
    e_data = (log_F_hat - log_F_obs_train) ** 2                       # (n, K)

    def ode_residual(tau):
        A_tau = grad(A_hat_fn)(tau)
        B = B_coeff(tau, p_Q.kappa)
        rhs = (p_Q.r
               + (p_Q.kappa * p_Q.alpha_Q + p_Q.rho * p_Q.sigma1 * p_Q.sigma2) * B
               + 0.5 * p_Q.sigma2 ** 2 * B ** 2)
        return A_tau - rhs

    e_ode = vmap(ode_residual)(tau_mesh) ** 2                         # (N_f,)


    dt = jnp.diff(t_train)                                            # (n-1,)

    # e_sde trains ONLY alpha_P: kappa/sigma2 are already identified by the pricing
    # channels, and an OU-MLE on the smoothed NN path is degenerate for them (drives
    # kappa->0). kappa_P/sigma2_P and the path itself are stop-gradiented here.
    kappa_P_sde = lax.stop_gradient(kappa_P)
    sigma2_P_sde = lax.stop_gradient(sigma2_P)

    def sde_nll(delta_i, delta_ip1, dt_i):
        mean = alpha_P + (delta_i - alpha_P) * jnp.exp(-kappa_P_sde * dt_i)
        var = (sigma2_P_sde ** 2 / (2.0 * kappa_P_sde)) * (1.0 - jnp.exp(-2.0 * kappa_P_sde * dt_i))
        return 0.5 * jnp.log(2.0 * jnp.pi * var) + 0.5 * (delta_ip1 - mean) ** 2 / var

    delta_hat_sde = lax.stop_gradient(delta_hat)
    e_sde = vmap(sde_nll)(delta_hat_sde[:-1], delta_hat_sde[1:], dt)  # (n-1,)

    # No-arb hinges (deriv-6 S7.1-7.2), on the data manifold -- same (n,K) grid as e_data,
    # reusing F_hat = exp(log_F_hat). cac ceiling excludes delta (S7.1); rcac floor carries
    # the delta_max wedge (S7.2). Hinge squared is C1 through the kink (eq. 49 remark).
    F_hat_grid = jnp.exp(log_F_hat)                                    # (n, K)
    ceiling = S_train[:, None] * jnp.exp((p_Q.r + STORAGE_COST_U) * taus)[None, :]
    rcac_floor = S_train[:, None] * jnp.exp((p_Q.r - DELTA_MAX) * taus)[None, :]
    e_cac = jnp.maximum(0., F_hat_grid - ceiling) ** 2                 # (n, K)
    e_rcac = jnp.maximum(0., rcac_floor - F_hat_grid) ** 2             # (n, K)

    err = {
        "e_data": e_data,
        "e_ode": e_ode,
        "e_sde": e_sde,
        "e_cac": e_cac,
        "e_rcac": e_rcac,
    }
    metrics = {k: jnp.mean(v) for k, v in err.items()}
    return err, metrics

In [ ]:
def adj(loss, lw):
    return lw * jnp.mean(loss)


def make_loss_lb(components):
    def loss_fn(fns, data, mesh, l_ws):
        err, metrics = error(fns, data, mesh)
        loss = {k: adj(err[k], l_ws[k]) for k in components}
        return loss, metrics
    return loss_fn


def make_loss_fn(components):
    def loss_fn(fns, data, mesh, l_ws):
        loss, metrics = loss_fn_lb[components](fns, data, mesh, l_ws)
        return sum(loss.values()), metrics
    return loss_fn


# single source of truth for which loss channels each variant trains on --
# init_l_ws (below) derives its channel sets from this too, instead of being a
# second, independently-typed list that has to be kept in sync by hand.
VARIANT_COMPONENTS = {
    "MLP": ["e_data"],                                    # data-misfit only
    "PINN": ["e_data", "e_ode"],                           # simple PINN
    "APPINN": ["e_data", "e_ode", "e_sde"],                # full AP-PINN architecture
    "APPINN_ARB": ["e_data", "e_ode", "e_sde", "e_cac", "e_rcac"],  # + no-arb hinges
}

loss_fn_lb = {k: make_loss_lb(v) for k, v in VARIANT_COMPONENTS.items()}
loss_fn = {k: make_loss_fn(k) for k in loss_fn_lb}

### Calibration

Training loop with WamOL gradient-norm balancing across the composite loss's components (`l_ws` for `e_data`/`e_ode`/`e_sde`), plus checkpointing. The per-point ("piecewise") self-adaptive weight optimizer from the original AC-PINN code (`params_sa`, SGD ascent) is removed -- only the component-level balancing is kept.

In [ ]:
def flatten_pytree(pytree):
    return ravel_pytree(pytree)[0]

# initial per-channel loss weights, derived from the same VARIANT_COMPONENTS
# that defines loss_fn_lb (cell above) -- one source of truth for "which
# channels does each variant train."
init_l_ws = {k: {c: 1. for c in v} for k, v in VARIANT_COMPONENTS.items()}

# Grad-norm balancing (S12.2, eq. 55-56): each component's norm uses only its own
# param support (e_data/e_cac/e_rcac: drift+path+psi, e_ode: drift+psi, e_sde: psi),
# not the full pytree -- avoids diluting the psi-only/drift-only channels with
# exact zeros from the params they don't touch. Module-level since it's a fixed
# constant independent of config/data -- calibration() below only reads it.
COMPONENT_SUPPORT = {
    "e_data": ("drift", "path", "psi"),
    "e_ode": ("drift", "psi"),
    "e_sde": ("psi",),  # NLL trains psi only (path grad stopped in error())
    "e_cac": ("drift", "path", "psi"),
    "e_rcac": ("drift", "path", "psi"),
}
FULL_SUPPORT = ("drift", "path", "psi")

# training loop
def calibration(config, data, mesh, hp=adam_hp):
    ofunc = loss_fn[config.loss_str]
    l_ws = dict(init_l_ws[config.loss_str])

    key = random.PRNGKey(config.seed)
    key, key_init = random.split(key, 2)

    # One TrainState over the joint pytree {"drift": theta, "path": phi} --
    drift_ann, path_ann, params_init = init_nets(config, key_init)
    tx = make_optimizer(hp)
    state = train_state.TrainState.create(apply_fn=None, params=params_init, tx=tx)

    @jit
    def train_step(state, data, mesh, l_ws):
        def loss_fn_(params):
            fns = make_fns(drift_ann, path_ann, params)
            return ofunc(fns, data, mesh, l_ws)

        (loss, metric), grads = jax.value_and_grad(loss_fn_, has_aux=True)(state.params)
        state = state.apply_gradients(grads=grads)
        return state, loss, metric

    hist_loss = []
    momentum = config.loss_balancing_momentum
    start_time = time.time()

    # grad_norm_full_support=True reverts to full-pytree norms for A/B comparison.
    use_full_grad_support = config.get("grad_norm_full_support", False)

    @jit
    def update_loss_weights(state, data, mesh, l_ws):
        def loss_fn_(params):
            fns = make_fns(drift_ann, path_ann, params)
            return loss_fn_lb[config.loss_str](fns, data, mesh, l_ws)[0]
        grads = jacrev(loss_fn_)(state.params)

        support = {k: FULL_SUPPORT for k in COMPONENT_SUPPORT} if use_full_grad_support else COMPONENT_SUPPORT
        grad_norm_dict = {
            k: jnp.abs(flatten_pytree({sk: v[sk] for sk in support[k]})).mean()
            for k, v in grads.items()
        }
        sum_grad_norm = jnp.sum(jnp.stack(tree_leaves(grad_norm_dict)))
        w = tree_map(lambda x: jnp.where(x == 0., 1., sum_grad_norm / x), grad_norm_dict)

        weights = tree_map(lambda old_w, new_w: old_w * momentum + (1 - momentum) * new_w, l_ws, w)
        return lax.stop_gradient(weights)

    # Training loop
    print(f"{config.loss_str} calibration------>")
    # Only the two full AP-PINN variants carry more than one loss channel and ever run
    # the balancer; balancing=False runs with fixed lambda=1 as a control.
    use_balancing = config.get("balancing", True) and config.loss_str in ("APPINN", "APPINN_ARB")
    for epoch in range(config.num_epochs):

        if use_balancing and epoch % 100 == 0:
            l_ws = update_loss_weights(state, data, mesh, l_ws)

        state, loss, metric = train_step(state, data, mesh, l_ws)

        # float(...) forces a device->host sync every epoch (needed for full-resolution curve);
        # done here once per channel, not deferred to plot time against a pickled JAX array.
        metric_snap = {k: float(v) for k, v in metric.items()}
        psi_snap = {k: float(v) for k, v in constrain_psi(state.params["psi"]).items()}
        hist_loss.append((epoch, float(loss), metric_snap, psi_snap))
        if (epoch % 1000) == 0:
            print(f"Epoch {epoch}: loss = {loss:.6f}", end="\r")

    comp_time = time.time() - start_time
    print(f"------> completed in {comp_time:.2f} seconds")

    # Save checkpoint
    CKPT_DIR = os.path.abspath(f'../data/output/checkpoints/{run_tag(config)}')
    ckpt = {'params': state.params, 'ls': l_ws}
    orbax_checkpointer = orbax.checkpoint.PyTreeCheckpointer()
    save_args = orbax_utils.save_args_from_target(ckpt)
    orbax_checkpointer.save(CKPT_DIR, ckpt, force=True, save_args=save_args)

    # downstream code needs the drift and the path callables separately
    return make_fns(drift_ann, path_ann, state.params), hist_loss

def run_tag(config):
    """File-safe tag: loss_str, with a `_nobal` suffix whenever balancing is off
    (not special-cased to one variant -- applies to any config with a balancing flag)."""
    suffix = "" if config.get("balancing", True) else "_nobal"
    return f"{config.loss_str}{suffix}"


def run_experiment(config, data, mesh, hp=adam_hp):
    fns, hist_loss = calibration(config, data, mesh, hp)

    results = {'config': config, 'history': hist_loss, 'adam_hp': dict(hp)}
    with open(f'../data/output/results/results_{run_tag(config)}.pkl', 'wb') as f:
        pickle.dump(results, f)

    return fns

### Run

Five variants run each time: `MLP` (data-misfit only), `PINN` (+ ODE residual), `AP-PINN` no-bal (`APPINN_nobal` — fixed `lambda=1`), `AP-PINN` (full architecture, WamOL balancing on), and `AP-PINN` with no-arb constraints (`APPINN_ARB` — adds `e_cac`/`e_rcac`, balancing on). `num_epochs=3000` was the smoke-test value; 40k is used below to let `psi` converge.

In [ ]:
config_MLP = ml_collections.ConfigDict({
    "loss_str": "MLP",
    "num_epochs": 40000,
    "drift_hidden_dim": (64, 64, 64, 64),
    "path_hidden_dim": (32, 32, 32),
    "ann_activation_str": "tanh",
    "loss_balancing_momentum": 0.5,
    "seed": 42,
    "ann_reparam": False,
    "balancing": True,     # WamOL whack-a-mole on by default (AP-PINN variants only)
    "grad_norm_full_support": False,  # True = A/B against old full-pytree grad-norm behavior
})

config_PINN = ml_collections.ConfigDict(config_MLP.to_dict())
config_PINN.loss_str = "PINN"

config_APPINN = ml_collections.ConfigDict(config_PINN.to_dict())
config_APPINN.loss_str = "APPINN"

# CONTROL: AP-PINN with the WamOL balancer OFF -- isolates whether the balancer drives any path collapse.
config_APPINN_nobal = ml_collections.ConfigDict(config_APPINN.to_dict())
config_APPINN_nobal.balancing = False

# AP-PINN + no-arbitrage hinge penalties (e_cac, e_rcac), balancing on.
config_APPINN_ARB = ml_collections.ConfigDict(config_APPINN.to_dict())
config_APPINN_ARB.loss_str = "APPINN_ARB"

# psi is a learned leaf in every run; train all five at 40k so it has time to move.
fns_MLP = run_experiment(config_MLP, data, mesh)
fns_PINN = run_experiment(config_PINN, data, mesh)
fns_APPINN_nobal = run_experiment(config_APPINN_nobal, data, mesh)
fns_APPINN = run_experiment(config_APPINN, data, mesh)
fns_APPINN_ARB = run_experiment(config_APPINN_ARB, data, mesh)

# single source of truth for the 5-variant roster -- every downstream eval/plot cell
# iterates this instead of re-listing the same (fns, config) pairing each time.
configs = [config_MLP, config_PINN, config_APPINN_nobal, config_APPINN, config_APPINN_ARB]
fns_by_tag = {
    run_tag(config_MLP): fns_MLP,
    run_tag(config_PINN): fns_PINN,
    run_tag(config_APPINN_nobal): fns_APPINN_nobal,
    run_tag(config_APPINN): fns_APPINN,
    run_tag(config_APPINN_ARB): fns_APPINN_ARB,
}

In [ ]:
# compute once per model, reused by this cell and the two eval cells below
eval_by_tag = {}
for tag, fns in fns_by_tag.items():
    _, delta_hat_fn, _, _ = fns
    err, metrics = error(fns, data, mesh)
    eval_by_tag[tag] = {"metrics": metrics, "delta_hat": vmap(delta_hat_fn)(t_train)}

for tag, ev in eval_by_tag.items():
    print(f"Metrics for {tag}:")
    for k, v in ev["metrics"].items():
        print(f"  {k}: {v:.6e}")


### Model Evals (MC-simulated data only — scores against `delta_true`/`PSI_TRUE`, no real-WTI equivalent)

In [ ]:
def sde_nll(delta_i, delta_ip1, dt_i, kappa, alpha, sigma):
    mean = alpha + (delta_i - alpha) * jnp.exp(-kappa * dt_i)
    var  = (sigma**2 / (2.0*kappa)) * (1.0 - jnp.exp(-2.0*kappa*dt_i))
    return 0.5*jnp.log(2.0*jnp.pi*var) + 0.5*(delta_ip1 - mean)**2 / var

def sde_floor(dt, kappa, sigma):
    var = (sigma**2 / (2.0*kappa)) * (1.0 - jnp.exp(-2.0*kappa*dt))
    return jnp.mean(0.5*jnp.log(2.0*jnp.pi*var))

In [ ]:
dt    = jnp.diff(t_train)
floor = sde_floor(dt, kappa_P, sigma2_P)
nll_v = vmap(sde_nll, in_axes=(0, 0, 0, None, None, None))
e_true = jnp.mean(nll_v(delta_true[:-1], delta_true[1:], dt,
                        kappa_P, alpha_P, sigma2_P))

print(f"floor (deterministic) {floor:+.4f}   true path {e_true:+.4f}\n")
for tag, ev in eval_by_tag.items():
    e = ev["metrics"]['e_sde']
    print(f"{tag:14s} e_sde {e:+.4f}  retained "
          f"{100*(e - floor)/(e_true - floor):5.1f}%")

In [ ]:
y = delta_true
sy = jnp.std(y)

for tag, ev in eval_by_tag.items():
    d = ev["delta_hat"]

    bias = jnp.mean(d - y)
    sd   = jnp.std(d)
    rho  = jnp.corrcoef(d, y)[0, 1]

    # MSE = bias^2 + (sd - sy)^2 + 2*sd*sy*(1 - rho)
    c_bias, c_amp, c_shape = bias**2, (sd - sy)**2, 2*sd*sy*(1 - rho)
    mse = jnp.mean((d - y)**2)

    print(f"{tag:14s} offset {bias:+.4f}  amp {sd/sy:.3f}  corr {rho:+.4f}")
    print(f"{'':14s} MSE {mse:.5f} = bias {c_bias:.5f} + amp {c_amp:.5f} "
          f"+ shape {c_shape:.5f}  (check {c_bias+c_amp+c_shape:.5f})")

### Visualization

Two composite plots compare all five models (`MLP`, `PINN`, `AP-PINN` no-bal, `AP-PINN`, `AP-PINN` no-arb): `plot_training_history_composite` (loss curves, now including `e_cac`/`e_rcac` panels) and `plot_delta_recovery_composite` (recovered vs. true $\delta_t$, RMSE in legend — the only place `delta_true` is used).

In [ ]:
ORDER = ["kappa", "sigma1", "sigma2", "rho", "alpha_Q", "alpha_P"]


def _load_history(tag):
    # psi_snapshot is history[3], added once psi became a learned leaf
    with open(f'../data/output/results/results_{tag}.pkl', 'rb') as f:
        return pickle.load(f)['history']


def plot_training_history_composite(configs, histories, save_path=None):
    """Per-channel losses (unweighted means), one panel per channel, all models overlaid."""
    channels = ['e_data', 'e_ode', 'e_sde', 'e_cac', 'e_rcac']
    yscale = {'e_data': 'log', 'e_ode': 'log', 'e_sde': 'linear', 'e_cac': 'log', 'e_rcac': 'log'}
    titles = {
        'e_data': r'$e_{\mathrm{data}}$ -- price misfit',
        'e_ode':  r'$e_{\mathrm{ODE}}$ -- drift residual',
        'e_sde':  r'$e_{\mathrm{SDE}}$ -- OU transition NLL',
        'e_cac':  r'$e_{\mathrm{cac}}$ -- cash-and-carry hinge',
        'e_rcac': r'$e_{\mathrm{rcac}}$ -- reverse cash-and-carry hinge',
    }
    runs = {}
    for config in configs:
        tag = run_tag(config)
        hist = histories[tag]
        epochs  = [h[0] for h in hist]
        metrics = [h[2] for h in hist]
        series  = {k: [m[k] for m in metrics] for k in metrics[0]}
        runs[tag] = (epochs, series, set(init_l_ws[config.loss_str]))

    fig, axes = plt.subplots(1, len(channels), figsize=(24, 4.5))
    for ax, ch in zip(axes, channels):
        for tag, (epochs, series, trained) in runs.items():
            if ch in trained and ch in series:
                ax.plot(epochs, series[ch], lw=1.3, label=tag)
        ax.set_xlabel('Epoch'); ax.set_ylabel(ch); ax.set_yscale(yscale[ch])
        ax.set_title(titles[ch]); ax.grid(True, which='both', alpha=0.3); ax.legend(fontsize=8)
    fig.suptitle('Per-channel training losses -- all models', y=1.02)
    fig.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show(); plt.close()


def plot_psi_trajectories(configs, histories, save_path=None):
    """Learned psi vs truth over training, one panel per parameter."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, p in zip(axes.ravel(), ORDER):
        for config in configs:
            tag = run_tag(config)
            hist = histories[tag]
            ax.plot([h[0] for h in hist], [h[3][p] for h in hist], lw=1.4, label=tag)
        ax.axhline(PSI_TRUE[p], color='black', ls='--', lw=1.2, label='true')
        ax.axhline(PSI_INIT[p], color='grey', ls=':', lw=1.0, label='init')
        ax.set_title(p); ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)
        if p == ORDER[0]:
            ax.legend(fontsize=8)
    fig.suptitle(r'Learned $\psi$ vs truth -- all models', y=1.01)
    fig.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show(); plt.close()


def print_psi_table(configs, histories):
    """Recovered psi vs true/init for each model (final epoch)."""
    tags = [run_tag(c) for c in configs]
    finals = {tag: histories[tag][-1][3] for tag in tags}
    head = f"{'param':>8} | {'true':>9} | {'init':>9} | " + " | ".join(f"{t:>9}" for t in tags)
    print(head); print('-' * len(head))
    for p in ORDER:
        row = f"{p:>8} | {PSI_TRUE[p]:9.4f} | {PSI_INIT[p]:9.4f} | " + \
              " | ".join(f"{finals[t][p]:9.4f}" for t in tags)
        print(row)


def plot_delta_recovery_composite(fns_by_tag, t, delta_true, save_path=None):
    """Overlay recovered delta_hat_phi(t) for all models vs the true (eval-only) path."""
    fig, ax = plt.subplots(1, 1, figsize=(9, 5))
    ax.plot(t, delta_true, color='black', lw=1.8, label=r'$\delta_t$ (true)')
    for tag, fns in fns_by_tag.items():
        _, delta_hat_fn, _, _ = fns
        delta_hat = vmap(delta_hat_fn)(t)
        rmse = float(jnp.sqrt(jnp.mean((delta_hat - delta_true) ** 2)))
        ax.plot(t, delta_hat, lw=1.2, ls='--', label=f'{tag} (RMSE {rmse:.4f})')
    ax.set_xlabel('t (years)'); ax.set_ylabel('Convenience yield'); ax.legend()
    ax.set_title('Recovered vs. true convenience yield -- all models')
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show(); plt.close()


# load each results pickle exactly once, reused by all three functions below
histories = {run_tag(c): _load_history(run_tag(c)) for c in configs}

print("Recovered psi vs true/init (final epoch):")
print_psi_table(configs, histories)

# Model comparison across loss-configuration variants is still in flux (this is where
# ACPINN/DCPINN naming has churned before -- see docs/project_summary.md). Render inline
# only, don't persist to figures/pinn/ -- the finalized comparison figures live in
# scripts/make_pinn_figures.py once a model is chosen as the thesis result.
plot_training_history_composite(configs, histories)
plot_psi_trajectories(configs, histories)
plot_delta_recovery_composite(fns_by_tag, t_train, delta_true)

## Multi-Path Verification for MC-simulated data only

In [ ]:
# taus/t_train/p_Q/kappa_P/alpha_P/sigma2_P aren't indexed by path (confirmed in
# get_data_mc) -- load the pickle once here and slice per path_id directly,
# instead of calling get_data_mc() 20x (each call reopens and unpickles the
# entire multi-path file just to read out one path's S/log_F_obs/delta_true).
with open(_MC_DATA_PATH, "rb") as f:
    _mc_data_all = pickle.load(f)

results = []

for path_id in range(20):
    S_train = _mc_data_all["S"][path_id]
    log_F_obs_train = _mc_data_all["log_F_obs"][path_id]
    delta_true = _mc_data_all["delta_true"][path_id]
    data = (t_train, S_train, log_F_obs_train)

    fns, hist_loss = calibration(config_APPINN, data, mesh, hp=adam_hp)
    _, metrics = error(fns, data, mesh)

    _, delta_hat_fn, _, gsp = fns
    delta_hat = vmap(delta_hat_fn)(t_train)
    delta_rmse = float(jnp.sqrt(jnp.mean((delta_hat - delta_true) ** 2)))

    # recovered psi for this path, straight off gsp -- no need to reload history
    psi_hat = psi_dict(gsp["p_Q"], gsp["alpha_P"])

    results.append({
        "path_id": path_id,
        "psi_hat": psi_hat,
        "delta_rmse": delta_rmse,
        "metrics": {k: float(v) for k, v in metrics.items()},
        # stored per-path so the composite plot below doesn't need to re-run calibration --
        # each path has its OWN delta_true/delta_hat, not a shared one
        "delta_true": np.asarray(delta_true),
        "delta_hat": np.asarray(delta_hat),
    })

    print(f"path {path_id:2d}  delta_rmse {delta_rmse:.4f}  "
          f"kappa_hat {psi_hat['kappa']:.3f}  alpha_P_hat {psi_hat['alpha_P']:.3f}")

In [ ]:
rmse_arr = np.array([r["delta_rmse"] for r in results])
print(f"delta RMSE across {len(results)} paths: mean {rmse_arr.mean():.4f}  std {rmse_arr.std():.4f}\n")

print(f"{'param':>8} | {'true':>9} | {'mean_hat':>9} | {'std_hat':>9}")
for p in ORDER:
    vals = np.array([r["psi_hat"][p] for r in results])
    print(f"{p:>8} | {PSI_TRUE[p]:9.4f} | {vals.mean():9.4f} | {vals.std():9.4f}")

In [ ]:
# plot delta recovery for each path - composite figure with subplots for each path
def plot_delta_recovery_per_path(results, t_train, save_path=None):
    n_paths = len(results)
    n_cols = 5
    n_rows = (n_paths + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3))
    axes = axes.flatten()

    for i, result in enumerate(results):
        path_id = result["path_id"]
        delta_rmse = result["delta_rmse"]

        ax = axes[i]
        # each path plotted against its OWN true/recovered delta, not a shared one
        ax.plot(t_train, result["delta_true"], color='black', lw=1.8, label=r'$\delta_t$ (true)')
        ax.plot(t_train, result["delta_hat"], lw=1.2, ls='--', label=f'$\hat{{\delta}}_t$ (RMSE {delta_rmse:.4f})')
        ax.set_title(f'Path {path_id} | RMSE {delta_rmse:.4f}')
        ax.set_xlabel('t (years)')
        ax.set_ylabel('Convenience yield')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    fig.suptitle('Recovered vs. true convenience yield per path', y=1.02)
    fig.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show(); plt.close()


plot_delta_recovery_per_path(results, t_train)  # synthetic-data ablation, inline only -- not a thesis figure